In [1]:
import multiprocessing
import os

In [2]:
import spacy
from spacy.util import compile_infix_regex
import spacy_alignments as tokenizations
from spacy.tokenizer import Tokenizer
from spacy.tokens import Doc, Token

In [3]:
from transformers import BertTokenizerFast, BertModel

In [4]:
from data import *

In [5]:
def create_custom_tokenizer(nlp, special_tokens):
    # Lấy phần tử tokenizer mặc định của spaCy
    infix_re = compile_infix_regex(nlp.Defaults.infixes)
    
    # Tạo một custom tokenizer từ spaCy mặc định
    custom_tokenizer = Tokenizer(nlp.vocab, infix_finditer=infix_re.finditer)

    # Thêm các token đặc biệt vào bộ tokenizer
    special_case = {token: [{spacy.symbols.ORTH: token}] for token in special_tokens}
    for token in special_tokens:
        custom_tokenizer.add_special_case(token, special_case[token])
    
    return custom_tokenizer

In [6]:
# Khởi tạo tokenizer và model BERT
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
# Thêm token đặc biệt nếu chưa có
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
num_added_toks = tokenizer.add_special_tokens(special_tokens)
bert.resize_token_embeddings(len(tokenizer))  # Resize embedding BERT
# logging.info("Info: BERT loaded")

# Tokenizer và DataLoader
spacy_nlp = spacy.load('en_core_web_sm')
spacy_nlp.tokenizer = create_custom_tokenizer(spacy_nlp, special_tokens['additional_special_tokens'])

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [7]:
data_name = 'TBD'

In [8]:
if data_name == 'i2b2':
    data_info = loadi2b2()
if data_name == 'MATRES':
    data_info = loadMATRES()
if data_name == 'TDDMan':
    data_info = loadTDDMan()
if data_name == 'TBD':
    data_info = loadTBD()

Label Mapping: {'AFTER': 0, 'BEFORE': 1, 'INCLUDES': 2, 'IS_INCLUDED': 3, 'SIMULTANEOUS': 4, 'VAGUE': 5}


In [9]:
train_df = data_info['train_df']
valid_df = data_info['valid_df']
test_df = data_info['test_df']
num_classes = data_info['num_classes']
relation_map = data_info['label_mapping']
id2label = data_info['id2label']

rule_filename = 'rules_TBD.txt'

In [10]:
print("Create dataloader:")
batch_size = 128
window_size = 128
MAX_LENGTH = 256
shuffle=False
n_processor = multiprocessing.cpu_count() - 1
print("batch_size: ", batch_size)
print("window_size: ", window_size)
print("MAX_LENGTH: ", MAX_LENGTH)

Create dataloader:
batch_size:  128
window_size:  128
MAX_LENGTH:  256


In [11]:
train_cache_path = f'/data/ddao/TRE/pretrained_models/Reasoning/{data_name}/cache/train/'
os.makedirs(train_cache_path, exist_ok=True)
train_token_cache_path = train_cache_path + "token_cache_bert.pt"
train_spacy_jsonl_path = train_cache_path + "spacy_cache_bert.jsonl"

test_cache_path = f'/data/ddao/TRE/pretrained_models/Reasoning/{data_name}/cache/test/'
os.makedirs(test_cache_path, exist_ok=True)
test_token_cache_path = test_cache_path + "token_cache_bert.pt"
test_spacy_jsonl_path = test_cache_path + "spacy_cache_bert.jsonl"

valid_cache_path = f'/data/ddao/TRE/pretrained_models/Reasoning/{data_name}/cache/valid/'
os.makedirs(valid_cache_path, exist_ok=True)
valid_token_cache_path = valid_cache_path + "token_cache_bert.pt"
valid_spacy_jsonl_path = valid_cache_path + "spacy_cache_bert.jsonl"

In [12]:
build_spacy_and_token_cache(
    df=train_df,              # df có cột: text, label_encoded, entity1_start/end/text, entity2_start/end/text, document_id
    spacy_nlp=spacy_nlp,
    tokenizer=tokenizer,
    window_size=window_size,
    max_length=MAX_LENGTH,
    jsonl_path=train_spacy_jsonl_path,
    token_pt=train_token_cache_path,
    batch_size=batch_size,
    n_process=n_processor
)

spaCy processing raw texts...
Done spaCy on raw texts.
Creating marked texts...
Created 4032 marked texts (after filtering).
spaCy processing marked texts...
Done spaCy on marked texts.
Saved cache:
 - /data/ddao/TRE/pretrained_models/Reasoning/TBD/cache/train/spacy_cache_bert.jsonl
 - /data/ddao/TRE/pretrained_models/Reasoning/TBD/cache/train/token_cache_bert.pt
Done in 88.38s


In [13]:
build_spacy_and_token_cache(
    df=test_df,              # df có cột: text, label_encoded, entity1_start/end/text, entity2_start/end/text, document_id
    spacy_nlp=spacy_nlp,
    tokenizer=tokenizer,
    window_size=window_size,
    max_length=MAX_LENGTH,
    jsonl_path=test_spacy_jsonl_path,
    token_pt=test_token_cache_path,
    batch_size=batch_size,
    n_process=n_processor
)

spaCy processing raw texts...
Done spaCy on raw texts.
Creating marked texts...
Created 1427 marked texts (after filtering).
spaCy processing marked texts...
Done spaCy on marked texts.
Saved cache:
 - /data/ddao/TRE/pretrained_models/Reasoning/TBD/cache/test/spacy_cache_bert.jsonl
 - /data/ddao/TRE/pretrained_models/Reasoning/TBD/cache/test/token_cache_bert.pt
Done in 25.98s


In [14]:
build_spacy_and_token_cache(
    df=valid_df,              # df có cột: text, label_encoded, entity1_start/end/text, entity2_start/end/text, document_id
    spacy_nlp=spacy_nlp,
    tokenizer=tokenizer,
    window_size=window_size,
    max_length=MAX_LENGTH,
    jsonl_path=valid_spacy_jsonl_path,
    token_pt=valid_token_cache_path,
    batch_size=batch_size,
    n_process=n_processor
)

spaCy processing raw texts...
Done spaCy on raw texts.
Creating marked texts...
Created 629 marked texts (after filtering).
spaCy processing marked texts...
Done spaCy on marked texts.
Saved cache:
 - /data/ddao/TRE/pretrained_models/Reasoning/TBD/cache/valid/spacy_cache_bert.jsonl
 - /data/ddao/TRE/pretrained_models/Reasoning/TBD/cache/valid/token_cache_bert.pt
Done in 19.45s
